# Dimensionality Reduction - PCA & t-SNE
### Digits Dataset | Module 2

## 1. PCA - Principal Component Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [ ]:
# loading digits dataset - 1797 samples, 64 features (8x8 pixel images)
digits = load_digits()
df = pd.DataFrame(digits.data)
df['label'] = digits.target
print(df.shape)
df.head()

In [ ]:
# lets see what the digit images look like
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'label: {digits.target[i]}')
    ax.axis('off')
plt.suptitle('Sample Digit Images')
plt.tight_layout()
plt.show()

In [ ]:
# separating features and labels
X = digits.data
y = digits.target

# scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('original shape:', X_scaled.shape)

In [ ]:
# how many components do we need?
# explained variance ratio tells us how much info each component holds
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(cumulative_var, 'b-o', markersize=4)
plt.axhline(y=0.95, color='red', linestyle='--', label='95% variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA - Explained Variance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

n_95 = np.argmax(cumulative_var >= 0.95) + 1
print(f'Components needed for 95% variance: {n_95}')

In [ ]:
# applying PCA to reduce to 2D for visualization
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled)

print('reduced shape:', X_pca.shape)
print(f'variance explained by 2 components: {sum(pca_2d.explained_variance_ratio_)*100:.1f}%')

In [ ]:
# visualizing PCA 2D
plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', alpha=0.7, s=30)
plt.colorbar(scatter, label='Digit')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% var)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% var)')
plt.title('PCA - 2D Projection of Digits')
plt.grid(True, alpha=0.3)
plt.show()
# PCA is linear so some digits overlap - thats expected

## 2. t-SNE - t-Distributed Stochastic Neighbor Embedding

In [ ]:
# t-SNE is non-linear, much better at separating clusters visually
# but it's slow on large datasets - thats why we first reduce with PCA
# common trick: PCA first then t-SNE

# step 1: reduce to 30 components with PCA first
pca_30 = PCA(n_components=30, random_state=42)
X_pca_30 = pca_30.fit_transform(X_scaled)
print('after PCA:', X_pca_30.shape)

In [ ]:
# step 2: apply t-SNE on the 30 PCA components
# perplexity = roughly how many neighbors to consider (5-50 is typical)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_pca_30)
print('t-SNE output shape:', X_tsne.shape)

In [ ]:
# visualizing t-SNE 2D
plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab10', alpha=0.7, s=30)
plt.colorbar(scatter, label='Digit')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('t-SNE - 2D Projection of Digits')
plt.grid(True, alpha=0.3)
plt.show()
# much better separation than PCA - you can clearly see 10 clusters

In [ ]:
# side by side comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sc1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', alpha=0.7, s=25)
axes[0].set_title('PCA (linear)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].grid(True, alpha=0.3)
plt.colorbar(sc1, ax=axes[0], label='Digit')

sc2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab10', alpha=0.7, s=25)
axes[1].set_title('t-SNE (non-linear)')
axes[1].set_xlabel('Component 1')
axes[1].set_ylabel('Component 2')
axes[1].grid(True, alpha=0.3)
plt.colorbar(sc2, ax=axes[1], label='Digit')

plt.suptitle('PCA vs t-SNE Comparison', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# saving reduced data
df_out = pd.DataFrame({
    'pca_1': X_pca[:, 0],
    'pca_2': X_pca[:, 1],
    'tsne_1': X_tsne[:, 0],
    'tsne_2': X_tsne[:, 1],
    'label': y
})
df_out.to_csv('../data/digits_reduced.csv', index=False)
print('saved to data/digits_reduced.csv')
df_out.head()

### Conclusion

- digits dataset has 64 features (8x8 pixel values) - way too many to visualize directly
- PCA reduced it to 2D linearly - some overlap between digits like 3,5,8
- t-SNE reduced it non-linearly - way cleaner separation, 10 clusters visible clearly
- used PCA → t-SNE pipeline to speed things up (common trick in practice)
- PCA is fast and interpretable, t-SNE is slow but visually much better
- use PCA when you need to keep components meaningful, t-SNE only for visualization